In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle("customer_analysis.pkl")
df.shape

(66041, 22)

# 05. 가설 검증

## 분석 기준

- 분석 단위는 고객이 프로모션을 수신한 건이다.
- 관찰 종료 시점 전에 유효기간이 끝난 66,041건만 사용한다.
- `offer received`는 수신, `offer viewed`는 열람, `offer completed`는 완료를 의미한다.
- 정보 제공형 프로모션은 완료 이벤트가 없으므로 완료율 검정에서 제외한다.
- 고객정보가 없는 2,175명은 성별·나이·소득 관련 검정에서만 제외한다.
- 성별 `O`는 유효한 `Other` 값이며, 여성과 남성만 비교하는 검정에서는 별도로 제외한다.
- 통계적으로 유의한 관계가 확인되어도 인과관계로 단정하지 않는다.

## 가설 1. 카페인 민감자의 커피 신메뉴 프로모션 선호도 불일치

### 검증 가능 여부

검증 불가

### 판단 근거

현재 데이터에는 고객의 카페인 민감도, 선호 메뉴 및 신메뉴 이용 여부에 관한 변수가 없다. 따라서 현재 데이터만으로는 해당 가설을 검증할 수 없다. 향후 설문조사나 메뉴별 구매 데이터를 추가로 확보해야 한다.

In [3]:
df.columns.tolist()

['customer_id',
 'offer_id',
 'received_time',
 'expiry_time',
 'viewed_time',
 'completed_time',
 'offer_type',
 'reward',
 'difficulty',
 'duration',
 'channels',
 'viewed',
 'completed',
 'viewed_before_completed',
 'completed_without_prior_view',
 'gender',
 'age',
 'became_member_on',
 'income',
 'gender_group',
 'income_group',
 'age_group']

## 가설 2. 소비 부담 대비 낮은 보상

### 가설

보상(`reward`)이 낮거나 달성 조건(`difficulty`)이 높을수록 열람 후 완료 가능성이 낮을 것이다.

### 검정 기준

- 대상: 프로모션을 열람한 BOGO·할인형 수신 건
- 결과변수: 열람 후 완료 여부(`viewed_before_completed`)
- 설명변수: 보상(`reward`), 달성 난이도(`difficulty`)
- 통제변수: 프로모션 유형(`offer_type`)
- 검정 방법: 로지스틱 회귀
- 동일 고객이 여러 프로모션을 받을 수 있으므로 고객 ID 기준 군집 표준오차를 적용한다.
- 현재 데이터는 관찰 데이터이므로 결과를 인과관계로 해석하지 않는다.

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# BOGO·할인형 중 실제로 열람한 수신 건만 선택
h2 = df[
    df["offer_type"].isin(["bogo", "discount"])
    & df["viewed"].eq(True)
].copy()

# 분석하기 쉬운 0/1 변수로 변환
h2["completed_after_view"] = (
    h2["viewed_before_completed"].astype(int)
)

# 고객별 반복 관측을 고려한 로지스틱 회귀
h2_model = smf.logit(
    "completed_after_view ~ reward + difficulty + C(offer_type)",
    data=h2
).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": h2["customer_id"]}
)

print("분석 건수:", len(h2))
print("열람 후 완료율:", round(h2["completed_after_view"].mean() * 100, 2), "%")
print(h2_model.summary())

분석 건수: 39470
열람 후 완료율: 50.69 %
                            Logit Regression Results                            
Dep. Variable:     completed_after_view   No. Observations:                39470
Model:                            Logit   Df Residuals:                    39466
Method:                             MLE   Df Model:                            3
Date:                  Thu, 10 Sep 2026   Pseudo R-squ.:                 0.02382
Time:                          17:39:08   Log-Likelihood:                -26703.
converged:                         True   LL-Null:                       -27355.
Covariance Type:                cluster   LLR p-value:                3.266e-282
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     0.4409      0.042     10.412      0.000       0.358       0.524
C(offer_type)[T.discount]     0.2936   

### 검정 결과

분석 대상은 BOGO·할인형 프로모션을 열람한 39,470건이며, 열람 후 완료율은 50.69%였다.

고객별 반복 관측을 고려한 로지스틱 회귀 결과, 달성 난이도의 계수는 -0.0113이고 p값은 0.020으로 나타났다. 따라서 난이도가 높을수록 열람 후 완료 가능성이 낮아지는 경향이 확인되어 해당 부분은 가설과 일치했다.

반면 보상의 계수는 -0.0764이고 p값은 0.001 미만으로, 예상과 달리 보상이 높을수록 완료 가능성이 낮게 나타났다. 할인형 프로모션의 계수는 0.2936이고 p값은 0.001 미만으로, 다른 조건을 통제했을 때 할인형이 BOGO보다 완료 가능성이 높은 것으로 나타났다.

따라서 ‘낮은 보상과 높은 난이도가 완료를 방해한다’는 가설은 부분적으로만 지지되었다. 난이도에 관한 예상은 결과와 일치했지만, 보상은 예상과 반대 방향으로 나타났다.

다만 프로모션 설계가 8개뿐이고 보상·난이도·유형이 서로 결합되어 있으므로, 각 변수의 독립적인 인과효과로 해석할 수 없다. 회귀모형의 설명력도 Pseudo R² 기준 약 0.024로 낮아 다른 요인의 추가 검토가 필요하다. A/B 테스트는 현재 데이터로 수행할 수 없으며 향후 실험 설계로 제안할 수 있다.

## 가설 3. 기존 구매 습관으로 인한 프로모션 외면

고객의 프로모션 수신 이전 거래 횟수와 결제금액을 계산하면 기존 구매 습관과 프로모션 열람 여부의 관계를 분석할 수 있다.

다만 거래 데이터에는 특정 결제가 특정 프로모션의 영향으로 발생했는지를 나타내는 프로모션 ID가 없다. 따라서 기존 구매 습관과 열람 여부의 연관성은 분석할 수 있지만, 프로모션이 구매 행동을 변화시켰는지는 직접 검증할 수 없다.

해당 가설은 이전 거래 횟수와 결제금액을 파생변수로 생성한 후 추가 검증한다.

## 가설 4. 채널 구성에 따른 프로모션 열람률 차이

### 가설

Mobile·Social 채널이 포함된 프로모션은 Web·Email 중심 프로모션보다 수신 후 열람률이 높을 것이다.

### 검증 기준

- 대상: 관찰기간이 확보된 전체 프로모션 수신 건
- 결과변수: 유효기간 내 열람 여부(`viewed`)
- 설명변수: Web·Email·Mobile·Social 채널 포함 여부
- 비교 방법: 채널 포함 여부에 따른 수신 후 열람률 비교
- 원본 데이터에는 고객의 채널별 실제 사용량이나 도달 여부가 없으므로, 채널의 ‘사용률’이 아니라 프로모션의 ‘채널 포함 여부’를 분석한다.

### 1차 검증: 수신 건 단위 비교

관찰기간이 확보된 프로모션 수신 66,041건 중 49,417건이 유효기간 내 열람되어 전체 수신 후 열람률은 74.83%였다.

Mobile 포함 프로모션의 열람률은 78.18%로, 미포함 프로모션의 35.11%보다 43.07%p 높았다. Social 포함 프로모션의 열람률은 91.37%로, 미포함 프로모션의 48.55%보다 42.82%p 높았다.

반면 Web 포함 프로모션의 열람률은 72.51%로, 미포함 프로모션의 83.44%보다 10.93%p 낮았다. Email은 모든 수신 건에 포함되어 있어 포함 여부에 따른 비교가 불가능했다.

따라서 Mobile·Social이 포함된 프로모션에서 상대적으로 높은 열람률이 관찰되었으며, Web이 포함된 프로모션에서는 열람률이 높게 나타나지 않았다. 이는 가설과 대체로 일치하는 초기 결과이다.

다만 하나의 프로모션에 여러 채널이 동시에 포함되어 있고 프로모션 설계가 10개뿐이므로, 특정 채널이 열람률을 높이거나 낮췄다고 단정할 수 없다. 본 결과는 채널의 독립적인 효과가 아니라 채널 조합과 열람률 사이의 연관성으로 해석한다.

In [5]:
# 채널 포함 여부 파생변수 생성
h4 = df.copy()

for channel in ["web", "email", "mobile", "social"]:
    h4[f"has_{channel}"] = h4["channels"].apply(
        lambda values: channel in values
    )

# 채널별 포함 여부에 따른 수신 건수와 열람률
h4_results = []

for channel in ["web", "email", "mobile", "social"]:
    column = f"has_{channel}"

    for included, group in h4.groupby(column):
        h4_results.append({
            "채널": channel,
            "포함_여부": "포함" if included else "미포함",
            "수신_건수": len(group),
            "열람_건수": int(group["viewed"].sum()),
            "수신후_열람률": round(group["viewed"].mean() * 100, 2)
        })

h4_summary = pd.DataFrame(h4_results)

display(h4_summary)

,채널,포함_여부,수신_건수,열람_건수,수신후_열람률
0,web,미포함,14000,11682,83.44
1,web,포함,52041,37735,72.51
2,email,포함,66041,49417,74.83
3,mobile,미포함,5133,1802,35.11
4,mobile,포함,60908,47615,78.18
5,social,미포함,25512,12386,48.55
6,social,포함,40529,37031,91.37


### 본 검증: Offer별 단위 비교

팀에서 정한 검증 기준에 맞춰, 수신 건 단위 결과에 더해 10개 Offer 각각의 수신 후 열람률을 계산하였다. 이후 각 채널의 포함 여부에 따라 Offer별 열람률의 평균을 비교하였다.

In [6]:
# 채널 조합을 문자열로 변환
h4["channel_combination"] = h4["channels"].apply(
    lambda values: ", ".join(values)
)

# 10개 Offer별 열람률 계산
h4_offer_summary = (
    h4.groupby(
        [
            "offer_id",
            "offer_type",
            "reward",
            "difficulty",
            "duration",
            "channel_combination"
        ],
        as_index=False
    )
    .agg(
        수신_건수=("offer_id", "size"),
        열람_건수=("viewed", "sum"),
        수신후_열람률=("viewed", lambda x: round(x.mean() * 100, 2))
    )
)

# Offer별 채널 포함 여부 생성
for channel in ["web", "email", "mobile", "social"]:
    h4_offer_summary[f"has_{channel}"] = (
        h4_offer_summary["channel_combination"]
        .str.split(", ")
        .apply(lambda values: channel in values)
    )

print("[Offer별 열람률]")
display(
    h4_offer_summary[
        [
            "offer_id",
            "offer_type",
            "channel_combination",
            "수신_건수",
            "열람_건수",
            "수신후_열람률"
        ]
    ].sort_values("수신후_열람률", ascending=False)
)

# 채널 포함 여부별 Offer 열람률의 단순 평균
offer_channel_results = []

for channel in ["web", "email", "mobile", "social"]:
    column = f"has_{channel}"

    for included, group in h4_offer_summary.groupby(column):
        offer_channel_results.append({
            "채널": channel,
            "포함_여부": "포함" if included else "미포함",
            "Offer_개수": len(group),
            "Offer별_평균_열람률": round(
                group["수신후_열람률"].mean(), 2
            )
        })

offer_channel_summary = pd.DataFrame(offer_channel_results)

print("[채널 포함 여부별 Offer 단위 비교]")
display(offer_channel_summary)

[Offer별 열람률]


,offer_id,offer_type,channel_combination,수신_건수,열람_건수,수신후_열람률
9,fafdcd668e3743c1bb461111dcafc2a4,discount,"web, email, mobile, social",5033,4869,96.74
1,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,"web, email, mobile, social",6332,6037,95.34
4,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,"web, email, mobile, social",7593,7235,95.29
8,f19421c1d4aa40978ebb69ca19b0e20d,bogo,"web, email, mobile, social",7571,7208,95.21
7,ae264e3637204a6fb9bb56bc8210ddfd,bogo,"email, mobile, social",6382,5533,86.70
5,5a8bc65990b245e5a138643cd4eb9837,informational,"email, mobile, social",7618,6149,80.72
6,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,"web, email, mobile",6351,3392,53.41
2,2906b810c7d4411798c6938adc9daaa5,discount,"web, email, mobile",6411,3394,52.94
3,3f207df678b143eea3cee63160fa8bed,informational,"web, email, mobile",7617,3798,49.86
0,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,"web, email",5133,1802,35.11


[채널 포함 여부별 Offer 단위 비교]


,채널,포함_여부,Offer_개수,Offer별_평균_열람률
0,web,미포함,2,83.71
1,web,포함,8,71.74
2,email,포함,10,74.13
3,mobile,미포함,1,35.11
4,mobile,포함,9,78.47
5,social,미포함,4,47.83
6,social,포함,6,91.67


### 검정 결과

관찰기간이 확보된 10개 Offer를 각각 집계하여 채널 포함 여부에 따른 평균 열람률을 비교하였다.

- Web 미포함 Offer 2개의 평균 열람률은 83.71%, 포함 Offer 8개는 71.74%였다.
- Mobile 미포함 Offer 1개의 열람률은 35.11%, 포함 Offer 9개의 평균 열람률은 78.47%였다.
- Social 미포함 Offer 4개의 평균 열람률은 47.83%, 포함 Offer 6개는 91.67%였다.
- Email은 10개 Offer에 모두 포함되어 있어 포함 여부에 따른 비교가 불가능했다.

Mobile과 Social이 포함된 Offer에서 평균 열람률이 더 높게 나타났으며, Web은 포함된 Offer의 평균 열람률이 오히려 낮게 나타났다. 따라서 Mobile·Social 채널이 열람률과 관련 있다는 초기 가설은 기술통계상 지지되었다.

다만 Mobile 미포함 Offer가 1개뿐이고 Email은 모든 Offer에 포함되어 있으며, 하나의 Offer에 여러 채널과 설계 조건이 동시에 적용되어 있다. 따라서 현재 결과만으로 특정 채널이 열람률을 높였다고 단정할 수 없으며, 채널 조합과 열람률 사이의 연관성으로 해석해야 한다.

## 가설 5. 짧은 유효기간으로 인한 프로모션 인지 제한

### 가설

프로모션 유효기간(`duration`)이 짧을수록 고객이 유효기간 내 프로모션을 열람할 가능성이 낮을 것이다.

### 검정 기준

- 대상: 관찰기간이 확보된 전체 프로모션 수신 건
- 결과변수: 유효기간 내 열람 여부(`viewed`)
- 설명변수: 유효기간(`duration`)
- 참고변수: 수신부터 열람까지 걸린 시간(`viewed_time - received_time`)
- 검정 방법: 점이연 상관분석 및 로지스틱 회귀
- 동일 고객의 반복 수신을 고려해 로지스틱 회귀에 고객 ID 기준 군집 표준오차를 적용한다.

In [7]:
from scipy.stats import pointbiserialr
import statsmodels.formula.api as smf

h5 = df.copy()
h5["viewed_int"] = h5["viewed"].astype(int)

# 열람된 건의 수신 후 열람 소요시간
h5["view_delay_hours"] = (
    h5["viewed_time"] - h5["received_time"]
)

# 유효기간과 열람 여부의 점이연 상관분석
h5_corr, h5_corr_p = pointbiserialr(
    h5["viewed_int"],
    h5["duration"]
)

# 유효기간에 따른 열람 여부 로지스틱 회귀
h5_model = smf.logit(
    "viewed_int ~ duration",
    data=h5
).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": h5["customer_id"]}
)

# 유효기간별 기초 결과
h5_summary = (
    h5.groupby("duration", as_index=False)
    .agg(
        수신_건수=("offer_id", "size"),
        열람_건수=("viewed_int", "sum"),
        수신후_열람률=("viewed_int", lambda x: round(x.mean() * 100, 2)),
        열람_소요시간_중앙값=("view_delay_hours", "median")
    )
)

print("상관계수:", round(h5_corr, 4))
print("상관분석 p값:", h5_corr_p)
display(h5_summary)
print(h5_model.summary())

상관계수: -0.0833
상관분석 p값: 4.081650522196554e-102


,duration,수신_건수,열람_건수,수신후_열람률,열람_소요시간_중앙값
0,3,7618,6149,80.72,12.0
1,4,7617,3798,49.86,18.0
2,5,15164,14443,95.25,12.0
3,7,25476,18356,72.05,18.0
4,10,10166,6671,65.62,18.0


                           Logit Regression Results                           
Dep. Variable:             viewed_int   No. Observations:                66041
Model:                          Logit   Df Residuals:                    66039
Method:                           MLE   Df Model:                            1
Date:                Thu, 10 Sep 2026   Pseudo R-squ.:                0.006110
Time:                        17:39:09   Log-Likelihood:                -37034.
converged:                       True   LL-Null:                       -37262.
Covariance Type:              cluster   LLR p-value:                5.048e-101
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.6514      0.030     55.381      0.000       1.593       1.710
duration      -0.0893      0.004    -20.363      0.000      -0.098      -0.081


### 검정 결과

전체 66,041건을 분석한 결과, 유효기간과 열람 여부의 상관계수는 -0.0833이고 p값은 0.001 미만으로 나타났다. 통계적으로 유의하지만 상관관계의 크기는 매우 작았다.

로지스틱 회귀에서도 유효기간의 계수는 -0.0893이고 p값은 0.001 미만으로 나타났다. 즉, 유효기간이 길수록 열람 가능성이 높아질 것이라는 가설과 반대로 유효기간이 길수록 열람 가능성이 낮아지는 방향이 확인되었다.

기간별 수신 후 열람률은 3일 80.72%, 4일 49.86%, 5일 95.25%, 7일 72.05%, 10일 65.62%로 나타났다. 열람률이 유효기간에 따라 일정하게 증가하거나 감소하지 않았으며, 5일 프로모션의 열람률이 가장 높았다.

열람된 프로모션의 수신 후 열람 소요시간 중앙값은 기간에 따라 12~18시간이었다. 이는 열람한 고객 대부분이 유효기간 종료보다 훨씬 이전에 프로모션을 확인했음을 보여준다.

따라서 ‘유효기간이 짧아 고객이 인지하기 전에 종료된다’는 가설은 현재 결과에서 지지되지 않았다. 다만 프로모션 기간은 채널·유형·보상 등 다른 설계요소와 함께 정해져 있으므로, 유효기간 자체가 열람률을 낮춘다는 인과관계로 해석할 수 없다.

## 가설 6. 반복적인 프로모션 노출에 따른 반응 둔화

### 가설

고객이 이전에 받은 프로모션 수가 많을수록 새로운 프로모션을 열람할 가능성이 낮을 것이다.

### 검정 기준

- 대상: 관찰기간이 확보된 전체 프로모션 수신 건
- 결과변수: 유효기간 내 열람 여부(`viewed`)
- 설명변수: 해당 수신 시점 이전에 고객이 받은 프로모션 수(`prior_offer_count`)
- 검정 방법: 로지스틱 회귀
- 동일한 시간에 수신된 프로모션은 서로를 이전 수신으로 계산하지 않는다.
- 동일 고객의 반복 수신을 고려해 고객 ID 기준 군집 표준오차를 적용한다.

In [8]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

h6 = df.copy()
h6["viewed_int"] = h6["viewed"].astype(int)

# 고객·수신 시각별 Offer 수 계산
receipt_counts = (
    h6.groupby(["customer_id", "received_time"])
    .size()
    .reset_index(name="offers_at_time")
    .sort_values(["customer_id", "received_time"])
)

# 현재 시각보다 앞서 수신한 Offer 수 계산
receipt_counts["prior_offer_count"] = (
    receipt_counts.groupby("customer_id")["offers_at_time"]
    .cumsum()
    - receipt_counts["offers_at_time"]
)

# 원래 분석 데이터에 연결
h6 = h6.merge(
    receipt_counts[
        ["customer_id", "received_time", "prior_offer_count"]
    ],
    on=["customer_id", "received_time"],
    how="left"
)

# 이전 수신 횟수 구간 생성
h6["prior_offer_group"] = np.select(
    [
        h6["prior_offer_count"].eq(0),
        h6["prior_offer_count"].eq(1),
        h6["prior_offer_count"].eq(2),
        h6["prior_offer_count"].eq(3)
    ],
    ["0회", "1회", "2회", "3회"],
    default="4회 이상"
)

# 구간별 열람률
h6_summary = (
    h6.groupby("prior_offer_group", as_index=False)
    .agg(
        수신_건수=("offer_id", "size"),
        열람_건수=("viewed_int", "sum"),
        수신후_열람률=("viewed_int", lambda x: round(x.mean() * 100, 2))
    )
)

# 이전 프로모션 수에 따른 열람 여부 검정
h6_model = smf.logit(
    "viewed_int ~ prior_offer_count",
    data=h6
).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": h6["customer_id"]}
)

display(h6_summary)
print(h6_model.summary())

,prior_offer_group,수신_건수,열람_건수,수신후_열람률
0,0회,16971,13007,76.64
1,1회,16691,12469,74.70
2,2회,15185,11012,72.52
3,3회,11010,8152,74.04
4,4회 이상,6184,4777,77.25


                           Logit Regression Results                           
Dep. Variable:             viewed_int   No. Observations:                66041
Model:                          Logit   Df Residuals:                    66039
Method:                           MLE   Df Model:                            1
Date:                Thu, 10 Sep 2026   Pseudo R-squ.:               5.997e-05
Time:                        17:39:10   Log-Likelihood:                -37259.
converged:                       True   LL-Null:                       -37262.
Covariance Type:              cluster   LLR p-value:                   0.03452
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             1.1125      0.014     77.107      0.000       1.084       1.141
prior_offer_count    -0.0143      0.007     -2.190      0.029      -0.027      -0.002


### 검정 결과

이전 프로모션 수신 횟수별 열람률은 0회 76.64%, 1회 74.70tls%, 2회 72.52%, 3회 74.04%, 4회 이상 77.25%로 나타났다. 0회부터 2회까지는 열람률이 감소했지만, 3회 이후에는 다시 증가해 일관된 하락 패턴은 확인되지 않았다.

고객별 반복 관측을 고려한 로지스틱 회귀 결과, 이전 프로모션 수의 계수는 -0.0143이고 p값은 0.029였다. 이전 수신 횟수가 1회 증가할 때 열람 오즈는 약 1.4% 감소하는 것으로 나타났다.

그러나 Pseudo R²는 약 0.00006으로 설명력이 매우 낮았으며, 4회 이상 집단의 열람률이 가장 높게 나타나는 등 구간별 결과도 선형적인 감소 형태가 아니었다.

따라서 반복 노출에 따라 반응이 둔화된다는 가설은 통계적으로 약한 관련성만 확인되었으며, 실질적인 영향은 매우 작다고 판단한다. 이 결과만으로 반복 노출이 고객 피로를 유발했다고 단정할 수 없다.

## 가설 7. 알림 해제 고객의 프로모션 노출 제한

원본 데이터에는 고객의 알림 허용 여부, 푸시 발송 성공 여부 및 실제 도달 여부가 포함되어 있지 않다. 따라서 알림 해제로 인해 프로모션 노출이 제한된다는 가설은 현재 데이터로 검증할 수 없다.

향후 앱 알림 설정 정보와 푸시 메시지 발송·도달 로그를 확보한 후 검증해야 한다.

## 가설 8. BOGO 유형의 달성 부담

### 가설

프로모션을 열람한 고객 중 BOGO 프로모션의 완료율은 할인형 프로모션보다 낮을 것이다.

### 검정 기준

- 대상: 프로모션을 열람한 BOGO·할인형 수신 건
- 결과변수: 열람 후 완료 여부(`viewed_before_completed`)
- 설명변수: 프로모션 유형(`offer_type`)
- 검정 방법: 카이제곱 검정 및 로지스틱 회귀
- 동일 고객의 반복 수신을 고려해 로지스틱 회귀에 고객 ID 기준 군집 표준오차를 적용한다.
- 데이터에는 고객이 느낀 수량 부담이 직접 기록되어 있지 않으므로, BOGO의 완료율이 낮더라도 그 원인이 수량 부담이라고 단정할 수 없다.

In [9]:
from scipy.stats import chi2_contingency
import pandas as pd
import statsmodels.formula.api as smf

# 열람한 BOGO·할인형 프로모션만 선택
h8 = df[
    df["offer_type"].isin(["bogo", "discount"])
    & df["viewed"].eq(True)
].copy()

# 열람 후 완료 여부를 0/1로 변환
h8["completed_after_view"] = (
    h8["viewed_before_completed"].astype(int)
)

# 유형별 열람 후 완료율
h8_summary = (
    h8.groupby("offer_type", as_index=False)
    .agg(
        열람_건수=("offer_id", "size"),
        열람후_완료_건수=("completed_after_view", "sum"),
        열람후_완료율=(
            "completed_after_view",
            lambda x: round(x.mean() * 100, 2)
        )
    )
)

# 카이제곱 검정
h8_table = pd.crosstab(
    h8["offer_type"],
    h8["completed_after_view"]
)

h8_chi2, h8_p, h8_dof, h8_expected = chi2_contingency(h8_table)

# 고객별 반복 관측을 고려한 로지스틱 회귀
h8_model = smf.logit(
    "completed_after_view ~ C(offer_type)",
    data=h8
).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": h8["customer_id"]}
)

display(h8_summary)
display(h8_table)

print("카이제곱 통계량:", round(h8_chi2, 4))
print("카이제곱 p값:", h8_p)
print(h8_model.summary())

,offer_type,열람_건수,열람후_완료_건수,열람후_완료율
0,bogo,23368,10317,44.15
1,discount,16102,9691,60.19


completed_after_view,0,1
offer_type,,
bogo,13051,10317
discount,6411,9691


카이제곱 통계량: 980.0053
카이제곱 p값: 3.9850469254098e-215
                            Logit Regression Results                            
Dep. Variable:     completed_after_view   No. Observations:                39470
Model:                            Logit   Df Residuals:                    39468
Method:                             MLE   Df Model:                            1
Date:                  Thu, 10 Sep 2026   Pseudo R-squ.:                 0.01802
Time:                          17:39:11   Log-Likelihood:                -26862.
converged:                         True   LL-Null:                       -27355.
Covariance Type:                cluster   LLR p-value:                2.094e-216
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -0.2351      0.016    -14.950      0.000      -0.266      -0.204
C(offer_type)[T.disco

### 검정 결과

프로모션을 열람한 BOGO·할인형 39,470건을 분석한 결과, BOGO의 열람 후 완료율은 44.15%, 할인형은 60.19%로 나타났다. 할인형이 BOGO보다 16.04%p 높았다.

카이제곱 검정 결과 통계량은 980.0053, p값은 0.001 미만으로, 프로모션 유형과 열람 후 완료 여부 사이에 통계적으로 유의한 관련성이 확인되었다.

고객별 반복 관측을 고려한 로지스틱 회귀에서도 할인형의 계수는 0.6483이고 p값은 0.001 미만이었다. 할인형의 열람 후 완료 오즈는 BOGO의 약 1.91배로 나타났다.

따라서 ‘BOGO의 열람 후 완료율이 할인형보다 낮을 것이다’라는 가설은 지지되었다. 다만 데이터에는 고객이 느끼는 수량 부담이 기록되어 있지 않으므로, 이러한 차이의 원인이 BOGO의 수량 부담이라고 단정할 수 없다.

## 가설 9. 높은 목표 금액에 따른 완료 부담

### 가설

프로모션의 달성 조건 금액(`difficulty`)이 높을수록 열람 후 완료 가능성이 낮을 것이다.

### 검정 기준

- 대상: 프로모션을 열람한 BOGO·할인형 수신 건
- 결과변수: 열람 후 완료 여부(`viewed_before_completed`)
- 설명변수: 달성 조건 금액(`difficulty`)
- 통제변수: 프로모션 유형(`offer_type`)
- 검정 방법: 로지스틱 회귀
- 동일 고객의 반복 수신을 고려해 고객 ID 기준 군집 표준오차를 적용한다.
- `difficulty`는 프로모션 완료에 필요한 금액이며, 고객이 실제로 느낀 소비 부담을 직접 측정한 값은 아니다.

In [10]:
import statsmodels.formula.api as smf

# 열람한 BOGO·할인형 프로모션만 선택
h9 = df[
    df["offer_type"].isin(["bogo", "discount"])
    & df["viewed"].eq(True)
].copy()

h9["completed_after_view"] = (
    h9["viewed_before_completed"].astype(int)
)

# 난이도별 열람 후 완료율
h9_summary = (
    h9.groupby("difficulty", as_index=False)
    .agg(
        열람_건수=("offer_id", "size"),
        열람후_완료_건수=("completed_after_view", "sum"),
        열람후_완료율=(
            "completed_after_view",
            lambda x: round(x.mean() * 100, 2)
        )
    )
)

# 프로모션 유형을 통제한 로지스틱 회귀
h9_model = smf.logit(
    "completed_after_view ~ difficulty + C(offer_type)",
    data=h9
).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": h9["customer_id"]}
)

display(h9_summary)
print(h9_model.summary())

,difficulty,열람_건수,열람후_완료_건수,열람후_완료율
0,5,10600,5323,50.22
1,7,6037,3684,61.02
2,10,21031,10025,47.67
3,20,1802,976,54.16


                            Logit Regression Results                            
Dep. Variable:     completed_after_view   No. Observations:                39470
Model:                            Logit   Df Residuals:                    39467
Method:                             MLE   Df Model:                            2
Date:                  Thu, 10 Sep 2026   Pseudo R-squ.:                 0.02193
Time:                          17:39:12   Log-Likelihood:                -26755.
converged:                         True   LL-Null:                       -27355.
Covariance Type:                cluster   LLR p-value:                3.281e-261
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     0.1343      0.029      4.574      0.000       0.077       0.192
C(offer_type)[T.discount]     0.7599      0.023     32.804      0.000 

### 검정 결과

프로모션을 열람한 BOGO·할인형 39,470건을 분석하였다. 열람 후 완료율은 difficulty 5에서 50.22%, 7에서 61.02%, 10에서 47.67%, 20에서 54.16%로 나타나 단순 집단 비교에서는 일관된 감소 형태가 나타나지 않았다.

프로모션 유형을 통제한 로지스틱 회귀 결과, difficulty의 계수는 -0.0479이고 p값은 0.001 미만이었다. 프로모션 유형이 같다는 조건에서 difficulty가 1단위 증가할 때 열람 후 완료 오즈는 약 4.7% 낮아지는 것으로 나타났다.

따라서 달성 조건 금액이 높을수록 열람 후 완료 가능성이 낮을 것이라는 가설은 통계적으로 지지되었다.

다만 difficulty별 완료율이 단순하게 계속 감소하지 않았으며, 분석 대상이 8개의 BOGO·할인형 Offer 설계로 구성되어 있어 difficulty가 보상·기간·채널 구성과 결합되어 있다. 따라서 높은 difficulty가 완료율 감소의 직접적인 원인이라고 단정할 수는 없다.

## 가설 10. 소득 대비 높은 달성 조건의 부담

### 가설

소득이 낮은 고객일수록 높은 달성 조건 금액(`difficulty`)이 열람 후 완료 가능성에 더 부정적인 영향을 줄 것이다.

### 검정 기준

- 대상: 프로모션을 열람한 BOGO·할인형 수신 건
- 고객정보가 없는 2,175명은 제외한다.
- 결과변수: 열람 후 완료 여부(`viewed_before_completed`)
- 설명변수: 소득(`income`), 달성 조건 금액(`difficulty`)
- 핵심 검정변수: 소득과 달성 조건 금액의 상호작용
- 통제변수: 프로모션 유형(`offer_type`)
- 검정 방법: 상호작용항을 포함한 로지스틱 회귀
- 동일 고객의 반복 수신을 고려해 고객 ID 기준 군집 표준오차를 적용한다.

In [11]:
import numpy as np
import statsmodels.formula.api as smf

# 열람한 BOGO·할인형 중 소득정보가 있는 건만 선택
h10 = df[
    df["offer_type"].isin(["bogo", "discount"])
    & df["viewed"].eq(True)
    & df["income"].notna()
].copy()

h10["completed_after_view"] = (
    h10["viewed_before_completed"].astype(int)
)

# 해석과 계산 안정성을 위해 소득을 1만 달러 단위로 변환
h10["income_10k"] = h10["income"] / 10000

# 8만 달러 기준 요약용 소득 집단
h10["income_80k_group"] = np.where(
    h10["income"] >= 80000,
    "8만 달러 이상",
    "8만 달러 미만"
)

# 소득집단·난이도별 완료율
h10_summary = (
    h10.groupby(
        ["income_80k_group", "difficulty"],
        as_index=False
    )
    .agg(
        열람_건수=("offer_id", "size"),
        열람후_완료_건수=("completed_after_view", "sum"),
        열람후_완료율=(
            "completed_after_view",
            lambda x: round(x.mean() * 100, 2)
        )
    )
)

# 소득과 difficulty의 상호작용 검정
h10_model = smf.logit(
    """
    completed_after_view
    ~ income_10k * difficulty
    + C(offer_type)
    """,
    data=h10
).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": h10["customer_id"]}
)

print("분석 건수:", len(h10))
display(h10_summary)
print(h10_model.summary())

분석 건수: 34097


,income_80k_group,difficulty,열람_건수,열람후_완료_건수,열람후_완료율
0,8만 달러 미만,5,6719,3537,52.64
1,8만 달러 미만,7,3942,2527,64.10
2,8만 달러 미만,10,13702,6688,48.81
3,8만 달러 미만,20,1041,660,63.40
4,8만 달러 이상,5,2383,1549,65.00
5,8만 달러 이상,7,1308,935,71.48
6,8만 달러 이상,10,4543,3006,66.17
7,8만 달러 이상,20,459,298,64.92


                            Logit Regression Results                            
Dep. Variable:     completed_after_view   No. Observations:                34097
Model:                            Logit   Df Residuals:                    34092
Method:                             MLE   Df Model:                            4
Date:                  Thu, 10 Sep 2026   Pseudo R-squ.:                 0.04128
Time:                          17:39:13   Log-Likelihood:                -22398.
converged:                         True   LL-Null:                       -23362.
Covariance Type:                cluster   LLR p-value:                     0.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -0.6304      0.115     -5.505      0.000      -0.855      -0.406
C(offer_type)[T.discount]     0.7804      0.026     30.312      0.000 

### 검정 결과

소득정보가 있는 BOGO·할인형 열람 건 34,097건을 분석하였다. 소득정보가 없는 5,373건은 이번 분석에서 제외하였다.

8만 달러 이상 고객의 열람 후 완료율은 모든 난이도 구간에서 8만 달러 미만 고객보다 높게 나타났다. 그러나 소득과 달성 조건 금액의 상호작용 계수는 0.0015, p값은 0.402로 통계적으로 유의하지 않았다.

따라서 ‘소득이 낮을수록 높은 달성 조건 금액의 부정적인 영향이 더 클 것이다’라는 가설은 현재 분석에서 지지되지 않았다. 고소득 고객의 전반적인 완료율은 높았지만, 소득에 따라 난이도의 영향 자체가 달라진다는 근거는 확인되지 않았다.

이 결과는 소득과 완료율의 연관성이 없다는 의미가 아니라, 소득과 난이도의 상호작용이 통계적으로 확인되지 않았다는 의미이다. 또한 관찰 데이터이므로 인과관계로 해석할 수 없다.

### 팀원 결과 참고

전체 difficulty를 포함한 분석에서는 소득과 difficulty의 상호작용이 통계적으로 유의하지 않아 가설이 지지되지 않았다(p=0.402).

다만 difficulty 20인 프로모션을 제외한 분석에서는 가설을 지지하는 결과가 나타날 수 있다. 그러나 difficulty 20은 데이터 오류가 아닌 유효한 프로모션이며, 단 하나의 Offer이면서 Mobile·Social 채널이 포함되지 않은 특이한 설계이다.

따라서 difficulty 20 제외 결과는 민감도 분석으로만 제시하며, 제외 여부에 따라 결론이 달라지므로 소득에 따라 difficulty의 영향이 다르다고 확정하기 어렵다.

## 가설 11. 고소득 고객에 대한 보상 금액의 동기부여 부족

### 가설

고소득 고객은 저소득 고객보다 보상 금액 증가에 따른 열람 후 완료율 상승 폭이 작을 것이다.

### 검정 기준

- 대상: 프로모션을 열람한 BOGO·할인형 수신 건
- 고객정보가 없는 2,175명은 제외한다.
- 결과변수: 열람 후 완료 여부(`viewed_before_completed`)
- 설명변수: 소득(`income`), 보상 금액(`reward`)
- 핵심 검정변수: 소득과 보상 금액의 상호작용
- 통제변수: 달성 조건 금액(`difficulty`), 프로모션 유형(`offer_type`)
- 검정 방법: 집단별 완료율 비교, 카이제곱 검정, 상호작용 로지스틱 회귀
- 보상 평균 4.2를 기준으로 평균 이하와 평균 초과 집단을 구분한다.
- 동일 고객의 반복 수신을 고려해 로지스틱 회귀에 고객 ID 기준 군집 표준오차를 적용한다.

In [12]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf

# 열람한 BOGO·할인형 중 소득정보가 있는 건만 선택
h11 = df[
    df["offer_type"].isin(["bogo", "discount"])
    & df["viewed"].eq(True)
    & df["income"].notna()
].copy()

h11["completed_after_view"] = (
    h11["viewed_before_completed"].astype(int)
)

# 소득과 보상 집단 생성
h11["income_group_test"] = np.where(
    h11["income"] >= 80000,
    "8만 달러 이상",
    "8만 달러 미만"
)

h11["reward_group"] = np.where(
    h11["reward"] > 4.2,
    "평균 초과",
    "평균 이하"
)

h11["income_10k"] = h11["income"] / 10000

# 소득·보상 집단별 완료율
h11_summary = (
    h11.groupby(
        ["income_group_test", "reward_group"],
        as_index=False
    )
    .agg(
        열람_건수=("offer_id", "size"),
        열람후_완료_건수=("completed_after_view", "sum"),
        열람후_완료율=(
            "completed_after_view",
            lambda x: round(x.mean() * 100, 2)
        )
    )
)

# 네 집단과 완료 여부의 카이제곱 검정
h11["income_reward_group"] = (
    h11["income_group_test"]
    + " / "
    + h11["reward_group"]
)

h11_table = pd.crosstab(
    h11["income_reward_group"],
    h11["completed_after_view"]
)

h11_chi2, h11_p, h11_dof, h11_expected = chi2_contingency(
    h11_table
)

# 소득과 보상의 상호작용 로지스틱 회귀
h11_model = smf.logit(
    """
    completed_after_view
    ~ income_10k * reward
    + difficulty
    + C(offer_type)
    """,
    data=h11
).fit(
    disp=False,
    cov_type="cluster",
    cov_kwds={"groups": h11["customer_id"]}
)

print("분석 건수:", len(h11))
display(h11_summary)
display(h11_table)

print("카이제곱 통계량:", round(h11_chi2, 4))
print("카이제곱 p값:", h11_p)
print(h11_model.summary())

분석 건수: 34097


,income_group_test,reward_group,열람_건수,열람후_완료_건수,열람후_완료율
0,8만 달러 미만,평균 이하,9177,5960,64.94
1,8만 달러 미만,평균 초과,16227,7452,45.92
2,8만 달러 이상,평균 이하,3196,2249,70.37
3,8만 달러 이상,평균 초과,5497,3539,64.38


completed_after_view,0,1
income_reward_group,,
8만 달러 미만 / 평균 이하,3217,5960
8만 달러 미만 / 평균 초과,8775,7452
8만 달러 이상 / 평균 이하,947,2249
8만 달러 이상 / 평균 초과,1958,3539


카이제곱 통계량: 1392.0102
카이제곱 p값: 1.5954780907693823e-301
                            Logit Regression Results                            
Dep. Variable:     completed_after_view   No. Observations:                34097
Model:                            Logit   Df Residuals:                    34091
Method:                             MLE   Df Model:                            5
Date:                  Thu, 10 Sep 2026   Pseudo R-squ.:                 0.04872
Time:                          17:39:13   Log-Likelihood:                -22224.
converged:                         True   LL-Null:                       -23362.
Covariance Type:                cluster   LLR p-value:                     0.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     0.6469      0.090      7.159      0.000       0.470       0.824
C(offer_type)[T.d

### 검정 결과

소득정보가 있는 BOGO·할인형 열람 건 34,097건을 분석하였다.

8만 달러 미만 고객의 열람 후 완료율은 보상 평균 이하 집단에서 64.94%, 평균 초과 집단에서 45.92%로, 높은 보상 집단에서 19.02%p 낮았다. 8만 달러 이상 고객은 평균 이하에서 70.37%, 평균 초과에서 64.38%로, 높은 보상 집단에서 5.99%p 낮았다.

카이제곱 검정 결과 통계량은 1,392.0102이고 p값은 0.001 미만으로, 소득·보상 집단과 열람 후 완료 여부 사이에 통계적으로 유의한 관련성이 확인되었다. 다만 카이제곱 검정은 네 집단 중 차이가 존재한다는 것만 보여주며, 소득에 따른 보상 효과의 차이는 상호작용항으로 판단해야 한다.

로지스틱 회귀에서 소득과 보상의 상호작용 계수는 0.0268이고 p값은 0.001 미만이었다. 예상과 달리 상호작용 계수가 양수로 나타나, 소득이 높을수록 보상 증가와 완료 가능성 사이의 부정적인 관계가 오히려 약해지는 것으로 나타났다.

따라서 ‘보상 금액이 고소득 고객의 동기부여 수준에 미달할 것이다’라는 가설은 지지되지 않았다. 오히려 높은 보상 집단에서 나타난 완료율 감소 폭은 고소득 고객보다 저소득 고객에게서 더 컸다.

다만 보상 수준은 프로모션 유형·난이도·기간·채널 구성과 결합되어 있으므로, 높은 보상 자체가 완료율을 낮춘다고 단정할 수 없다. 또한 고객의 동기부여 수준을 직접 측정한 데이터가 없으므로 완료 반응의 차이로만 해석해야 한다.

## 가설 12. Web·Email 중심 채널의 낮은 프로모션 반응

### 가설

Mobile·Social이 포함되지 않은 Web·Email 중심 프로모션은 Mobile·Social이 포함된 프로모션보다 고객의 반응률이 낮을 것이다.

### 검정 기준

- 대상: 관찰기간이 확보된 전체 프로모션 수신 건
- 결과변수: 유효기간 내 열람 여부(`viewed`)
- 설명변수: 채널 조합(`channel_combination`), Mobile 포함 여부, Social 포함 여부
- 검정 방법: 카이제곱 검정
- 원본 데이터에는 발송 실패나 실제 도달 여부가 없으므로, 발송 성공률이 아니라 `viewed`로 기록된 고객 반응을 비교한다.
- Email은 모든 Offer에 포함되어 있어 Email 단독 효과는 검정할 수 없다.

In [13]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

h12 = df.copy()
h12["viewed_int"] = h12["viewed"].astype(int)

# 채널 조합과 Mobile·Social 포함 여부 생성
h12["channel_combination"] = h12["channels"].apply(
    lambda values: ", ".join(values)
)

h12["has_mobile"] = h12["channels"].apply(
    lambda values: "mobile" in values
)

h12["has_social"] = h12["channels"].apply(
    lambda values: "social" in values
)

# 채널 조합별 열람률
h12_summary = (
    h12.groupby("channel_combination", as_index=False)
    .agg(
        수신_건수=("offer_id", "size"),
        열람_건수=("viewed_int", "sum"),
        수신후_열람률=(
            "viewed_int",
            lambda x: round(x.mean() * 100, 2)
        )
    )
    .sort_values("수신후_열람률", ascending=False)
)

# 채널 조합과 열람 여부의 카이제곱 검정
h12_table = pd.crosstab(
    h12["channel_combination"],
    h12["viewed_int"]
)

h12_chi2, h12_p, h12_dof, h12_expected = chi2_contingency(
    h12_table
)

# 효과크기: Cramér's V
h12_n = h12_table.to_numpy().sum()
h12_min_dim = min(h12_table.shape[0] - 1, h12_table.shape[1] - 1)

h12_cramers_v = np.sqrt(
    h12_chi2 / (h12_n * h12_min_dim)
)

display(h12_summary)
display(h12_table)

print("전체 분석 건수:", len(h12))
print("카이제곱 통계량:", round(h12_chi2, 4))
print("자유도:", h12_dof)
print("p값:", h12_p)
print("Cramér's V:", round(h12_cramers_v, 4))

,channel_combination,수신_건수,열람_건수,수신후_열람률
3,"web, email, mobile, social",26529,25349,95.55
0,"email, mobile, social",14000,11682,83.44
2,"web, email, mobile",20379,10584,51.94
1,"web, email",5133,1802,35.11


viewed_int,0,1
channel_combination,,
"email, mobile, social",2318,11682
"web, email",3331,1802
"web, email, mobile",9795,10584
"web, email, mobile, social",1180,25349


전체 분석 건수: 66041
카이제곱 통계량: 16570.2715
자유도: 3
p값: 0.0
Cramér's V: 0.5009


### 검정 결과

관찰기간이 확보된 프로모션 수신 66,041건을 채널 조합별로 비교하였다.

수신 후 열람률은 Web·Email·Mobile·Social 조합이 95.55%로 가장 높았고, Email·Mobile·Social 조합은 83.44%였다. Web·Email·Mobile 조합은 51.94%였으며, Mobile과 Social이 모두 포함되지 않은 Web·Email 조합은 35.11%로 가장 낮았다.

카이제곱 검정 결과 통계량은 16,570.2715, 자유도는 3, p값은 0.001 미만으로 나타났다. 따라서 채널 조합과 프로모션 열람 여부 사이에는 통계적으로 유의한 관련성이 확인되었다. Cramér’s V는 0.5009로, 관련성의 크기도 비교적 크게 나타났다.

Mobile과 Social이 모두 없는 Web·Email 조합의 열람률이 가장 낮고, Social이 포함된 두 채널 조합의 열람률이 상대적으로 높아 가설은 지지되었다.

다만 프로모션 설계가 10개뿐이고 각 채널 조합에 보상·난이도·기간·유형이 함께 결합되어 있다. 또한 동일 고객이 여러 번 포함되어 있으므로, 카이제곱 검정 결과만으로 특정 채널의 독립적인 인과효과를 단정할 수 없다. 실제 발송 성공이나 메시지 도달 여부도 없으므로 채널 조합과 열람 반응의 연관성으로 해석한다.

## 가설 13. 고령 고객의 프로모션 반응 저하

### 가설

고객의 나이가 많을수록 프로모션을 유효기간 내 열람할 가능성이 낮을 것이다.

### 검정 기준

- 대상: 관찰기간이 확보되고 유효한 나이 정보가 있는 전체 프로모션 수신 건
- 나이가 118세로 입력된 고객은 나이 관련 분석에서 제외한다.
- 결과변수: 유효기간 내 열람 여부(`viewed`)
- 설명변수: 고객 나이(`age`)
- 구분변수: 채널 조합(`channel_combination`)
- 검정 방법: 전체 및 채널 조합별 점이연 상관분석
- 현재 데이터는 프로모션을 수신한 기록으로 구성되어 있으므로 Offer 수신 자체가 아니라 수신 후 열람 반응을 분석한다.

In [14]:
import pandas as pd
from scipy.stats import pointbiserialr

# 유효한 나이 정보가 있는 수신 건만 선택
h13 = df[
    df["age"].notna()
    & df["age"].ne(118)
].copy()

h13["viewed_int"] = h13["viewed"].astype(int)

h13["channel_combination"] = h13["channels"].apply(
    lambda values: ", ".join(values)
)

h13["age_group_test"] = pd.cut(
    h13["age"],
    bins=[17, 54, float("inf")],
    labels=["54세 이하", "55세 이상"]
)

# 연령 집단별 열람률
h13_summary = (
    h13.groupby("age_group_test", observed=True, as_index=False)
    .agg(
        수신_건수=("offer_id", "size"),
        열람_건수=("viewed_int", "sum"),
        수신후_열람률=(
            "viewed_int",
            lambda x: round(x.mean() * 100, 2)
        )
    )
)

# 전체 나이와 열람 여부의 점이연 상관분석
h13_corr, h13_p = pointbiserialr(
    h13["viewed_int"],
    h13["age"]
)

# 채널 조합별 상관분석
h13_channel_results = []

for channel_name, group in h13.groupby("channel_combination"):
    corr, p_value = pointbiserialr(
        group["viewed_int"],
        group["age"]
    )

    h13_channel_results.append({
        "채널_조합": channel_name,
        "수신_건수": len(group),
        "상관계수": round(corr, 4),
        "p값": p_value
    })

h13_channel_summary = pd.DataFrame(h13_channel_results)

print("전체 분석 건수:", len(h13))
display(h13_summary)

print("전체 상관계수:", round(h13_corr, 4))
print("전체 p값:", h13_p)

display(h13_channel_summary)

전체 분석 건수: 57561


,age_group_test,수신_건수,열람_건수,수신후_열람률
0,54세 이하,27626,20444,74.00
1,55세 이상,29935,22238,74.29


전체 상관계수: 0.0201
전체 p값: 1.3568875794143707e-06


,채널_조합,수신_건수,상관계수,p값
0,"email, mobile, social",12197,-0.1168,2.633762e-38
1,"web, email",4509,0.0768,2.457503e-07
2,"web, email, mobile",17757,0.0893,9.284389e-33
3,"web, email, mobile, social",23098,0.0127,5.303406e-02


### 검정 결과

유효한 나이 정보가 있는 프로모션 수신 57,561건을 분석하였다. 54세 이하 고객의 수신 후 열람률은 74.00%, 55세 이상 고객은 74.29%로 나타나 55세 이상 집단이 0.29%p 높았다.

전체 나이와 열람 여부의 점이연 상관계수는 0.0201이고 p값은 0.001 미만이었다. 통계적으로는 유의하지만 상관계수가 0에 매우 가까워 실질적인 관계는 거의 없는 것으로 판단된다. 또한 상관관계의 방향도 예상과 달리 약한 양의 방향이었다.

채널 조합별 결과에서는 Email·Mobile·Social 조합만 나이와 열람 여부 사이에 음의 상관관계가 나타났다(r=-0.1168, p<0.001). Web·Email 조합은 r=0.0768, Web·Email·Mobile 조합은 r=0.0893으로 양의 관계가 나타났다. 네 채널이 모두 포함된 조합은 r=0.0127, p=0.053으로 통계적으로 유의하지 않았다.

따라서 ‘고령 고객일수록 프로모션 반응이 낮을 것이다’라는 가설은 전체 결과에서 지지되지 않았다. 채널 조합에 따라 관계의 방향이 달라지므로, 특정 채널에서 나타난 결과를 전체 고령 고객의 특성으로 일반화할 수 없다.

## 가설 14. 성별과 프로모션 유형에 따른 완료 반응 차이

### 가설

고객의 성별에 따라 BOGO와 할인형 프로모션의 열람 후 완료율이 다르게 나타날 것이다.

### 검정 기준

- 대상: 프로모션을 열람한 BOGO·할인형 수신 건
- 성별 정보가 없는 고객은 제외한다.
- 팀 분석 기준에 따라 성별 `O` 고객 212명은 성별 비교에서 제외하고 여성(F)과 남성(M)만 비교한다.
- 결과변수: 열람 후 완료 여부(`viewed_before_completed`)
- 설명변수: 성별(`gender`), 프로모션 유형(`offer_type`)
- 검정 방법: 성별·프로모션 유형 집단과 완료 여부의 카이제곱 검정
- 여기서 수용도는 직접 측정된 선호도가 아니라 열람 후 완료율로 정의한다.

In [15]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# 열람한 BOGO·할인형 중 성별이 F 또는 M인 건만 선택
h14 = df[
    df["offer_type"].isin(["bogo", "discount"])
    & df["viewed"].eq(True)
    & df["gender"].isin(["F", "M"])
].copy()

h14["completed_after_view"] = (
    h14["viewed_before_completed"].astype(int)
)

# 성별·프로모션 유형 결합 집단
h14["gender_offer_group"] = (
    h14["gender"] + " / " + h14["offer_type"]
)

# 집단별 열람 후 완료율
h14_summary = (
    h14.groupby(
        ["gender", "offer_type"],
        as_index=False
    )
    .agg(
        열람_건수=("offer_id", "size"),
        열람후_완료_건수=("completed_after_view", "sum"),
        열람후_완료율=(
            "completed_after_view",
            lambda x: round(x.mean() * 100, 2)
        )
    )
)

# 네 집단과 완료 여부의 카이제곱 검정
h14_table = pd.crosstab(
    h14["gender_offer_group"],
    h14["completed_after_view"]
)

h14_chi2, h14_p, h14_dof, h14_expected = chi2_contingency(
    h14_table
)

# 효과크기: Cramér's V
h14_n = h14_table.to_numpy().sum()
h14_min_dim = min(
    h14_table.shape[0] - 1,
    h14_table.shape[1] - 1
)

h14_cramers_v = np.sqrt(
    h14_chi2 / (h14_n * h14_min_dim)
)

print("분석 건수:", len(h14))
display(h14_summary)
display(h14_table)

print("카이제곱 통계량:", round(h14_chi2, 4))
print("자유도:", h14_dof)
print("p값:", h14_p)
print("Cramér's V:", round(h14_cramers_v, 4))

분석 건수: 33572


,gender,offer_type,열람_건수,열람후_완료_건수,열람후_완료율
0,F,bogo,8395,4916,58.56
1,F,discount,5779,4064,70.32
2,M,bogo,11536,4934,42.77
3,M,discount,7862,4942,62.86


completed_after_view,0,1
gender_offer_group,,
F / bogo,3479,4916
F / discount,1715,4064
M / bogo,6602,4934
M / discount,2920,4942


카이제곱 통계량: 1473.8673
자유도: 3
p값: 0.0
Cramér's V: 0.2095


### 검정 결과

성별이 F 또는 M으로 확인된 BOGO·할인형 열람 건 33,572건을 분석하였다.

여성 고객의 열람 후 완료율은 BOGO 58.56%, 할인형 70.32%였으며, 남성 고객은 BOGO 42.77%, 할인형 62.86%였다. 네 집단 중 여성·할인형의 완료율이 가장 높았고 남성·BOGO의 완료율이 가장 낮았다.

여성의 완료율은 남성보다 BOGO에서 15.79%p, 할인형에서 7.46%p 높았다. 또한 할인형은 BOGO보다 여성에게서 11.76%p, 남성에게서 20.09%p 높은 완료율을 보였다.

카이제곱 검정 결과 통계량은 1,473.8673, 자유도는 3, p값은 0.001 미만이었다. 따라서 성별·프로모션 유형 집단과 열람 후 완료 여부 사이에는 통계적으로 유의한 관련성이 확인되었다. Cramér’s V는 0.2095로 나타났다.

따라서 성별과 프로모션 유형에 따라 열람 후 완료율에 차이가 있을 것이라는 가설은 지지되었다. 특히 남성 고객의 BOGO 완료율이 상대적으로 낮게 나타났다.

다만 완료율 차이가 특정 프로모션에 대한 선호도나 수용도를 직접 의미하는 것은 아니다. 소득·나이·보상·난이도·채널 등의 차이가 함께 영향을 주었을 수 있으며, 관찰 데이터이므로 성별이 완료율 차이의 원인이라고 단정할 수 없다.

In [16]:
# 최종 숫자 검수
assert len(df) == 66041
assert int(df["viewed"].sum()) == 49417

assert len(h8) == 39470
assert int(h8["completed_after_view"].sum()) == 20008

assert len(h10) == 34097
assert len(h11) == 34097
assert len(h13) == 57561
assert len(h14) == 33572

print("최종 검수 완료: 주요 분석 건수와 결과가 모두 일치합니다.")

최종 검수 완료: 주요 분석 건수와 결과가 모두 일치합니다.
